In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
import sys 
sys.path.append("../")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 10

# Load the SVHN dataset
train_ds = torchvision.datasets.SVHN(root='./data', split='train', download=True)
test_ds = torchvision.datasets.SVHN(root='./data', split='test', download=True)

print(f"Training samples: {len(train_ds)}, Test samples: {len(test_ds)}")

mean = torch.tensor([0.4377, 0.4438, 0.4728])
std = torch.tensor([0.1980, 0.2010, 0.1970])

Training samples: 73257, Test samples: 26032


In [11]:
norm_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

weak_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

In [12]:
from datasets import TransformedDataset
from wideresnet2 import WideResNet 
from utils import evaluate_f1_and_accuracy
import os
import json
import time

num_runs = 3

test_ds = TransformedDataset(test_ds, norm_transform)

for run in range(num_runs):
    print(f"Run {run+1}/{num_runs}")

    # Create datasets and dataloaders
    labeled_ds = TransformedDataset(train_ds, weak_transform)

    # Create dataloaders
    batch_size = 64
    labeled_loader = DataLoader(labeled_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # Create iterators for the dataloaders
    labeled_iter = iter(labeled_loader)

    # Define the model
    model = WideResNet(depth=28, widen_factor=2, num_classes=num_classes).to(device)
    max_steps = 45_800  # Equivalent to 100 epochs on the full dataset with batch size 64
    optimizer = torch.optim.SGD(model.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)

    # Settings
    method_name = "Full_Supervised"
    name_of_experiment = f"svhn_full_sup_run_{run+1}"

    if os.path.exists(f"results/{name_of_experiment}/{method_name}.json"):
        print(f"Results for {name_of_experiment} already exist. Skipping saving to avoid overwriting.")
        break # this will skip the rest of the training loop and move to the next run

    # Metrics to track
    metrics = {
    "test_f1": [0.0],  # Start with 0% F1 before training
    "test_acc": [0.0],  # Start with 0% accuracy before training
    "budget": [0]
    }

    # Hyperparameters
    budget_per_iteration = 1  # 1 batch of labeled data per iteration
    test_budget_period = 900  # Evaluate on test set every 900 batches seen

    # Training loop
    current_budget = 0
    start_time = time.time()
    for step in range(max_steps):
        running_loss = 0.0
        running_loss_sup = 0.0
        model.train()
        try:
            x_l, y_l = next(labeled_iter)
        except StopIteration:
            labeled_iter = iter(labeled_loader)
            x_l, y_l = next(labeled_iter)

        x_l, y_l = x_l.to(device), y_l.to(device)

        # supervised
        logits_l = model(x_l)
        loss_sup = F.cross_entropy(logits_l, y_l)

        loss = loss_sup
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        running_loss_sup += loss_sup.item()

        current_budget += budget_per_iteration

        if current_budget % test_budget_period < budget_per_iteration:
            f1, acc = evaluate_f1_and_accuracy(model, test_loader, device)
            metrics["test_f1"].append(f1)
            metrics["test_acc"].append(acc)
            metrics["budget"].append(current_budget)
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Test F1: {f1:.4f}, Test Acc: {acc:.4f}, Elapsed Time: {elapsed_time:.2f}s", flush=True)

            os.makedirs("results", exist_ok=True)
            os.makedirs(f"results/{name_of_experiment}", exist_ok=True)
            with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                json.dump(metrics, f, indent=4)

        elif step+1 == max_steps: # Final evaluation at the end of training
            f1, acc = evaluate_f1_and_accuracy(model, test_loader, device)
            metrics["test_f1"].append(f1)
            metrics["test_acc"].append(acc)
            metrics["budget"].append(current_budget)
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Test F1: {f1:.4f}, Test Acc: {acc:.4f}, Elapsed Time: {elapsed_time:.2f}s", flush=True)

            os.makedirs("results", exist_ok=True)
            os.makedirs(f"results/{name_of_experiment}", exist_ok=True)
            with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                json.dump(metrics, f, indent=4) 

        else:
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}", end="\r", flush=True)

Run 1/3
Step 900/45800, Budget: 900, Loss: 0.4434, Sup Loss: 0.4434, Test F1: 0.8307, Test Acc: 0.8253, Elapsed Time: 26.34s
Step 1800/45800, Budget: 1800, Loss: 0.3798, Sup Loss: 0.3798, Test F1: 0.8916, Test Acc: 0.8920, Elapsed Time: 51.92s
Step 2700/45800, Budget: 2700, Loss: 0.2537, Sup Loss: 0.2537, Test F1: 0.9081, Test Acc: 0.9077, Elapsed Time: 77.84s
Step 3600/45800, Budget: 3600, Loss: 0.3771, Sup Loss: 0.3771, Test F1: 0.9095, Test Acc: 0.9097, Elapsed Time: 103.93s
Step 4500/45800, Budget: 4500, Loss: 0.3817, Sup Loss: 0.3817, Test F1: 0.9276, Test Acc: 0.9276, Elapsed Time: 129.33s
Step 5400/45800, Budget: 5400, Loss: 0.1876, Sup Loss: 0.1876, Test F1: 0.9070, Test Acc: 0.9075, Elapsed Time: 155.19s
Step 6300/45800, Budget: 6300, Loss: 0.5897, Sup Loss: 0.5897, Test F1: 0.9217, Test Acc: 0.9218, Elapsed Time: 181.26s
Step 7200/45800, Budget: 7200, Loss: 0.1825, Sup Loss: 0.1825, Test F1: 0.9301, Test Acc: 0.9299, Elapsed Time: 207.72s
Step 8100/45800, Budget: 8100, Loss: 